In [9]:
import requests
import os
from typing import Dict, Optional

class PrecisionCardPricer:
    def __init__(self):
        self.price_api_key = os.getenv("POKE_PRICE_TRACKER_API_KEY")
        self.api_calls_made = 0
        
    def get_exact_card_from_pokemontcg(self, card_input: str) -> Optional[Dict]:
        """
        Step 1: Get EXACT card details from pokemontcg.io (FREE)
        
        Example input: "Rayquaza V 194/203"
        Returns: Complete card data with exact names, set info, images
        """
        print(f"🔍 STEP 1: Finding exact card details for '{card_input}'")
        
        # Parse input to extract name and number
        parts = card_input.strip().split()
        
        # Handle "Name Number/Total" format
        if len(parts) >= 2 and '/' in parts[-1]:
            card_number = parts[-1].split('/')[0]  # "194" from "194/203"
            card_name = ' '.join(parts[:-1])       # "Rayquaza V"
        else:
            card_name = card_input
            card_number = None
        
        print(f"   Parsed - Name: '{card_name}', Number: '{card_number}'")
        
        # Build precise pokemontcg.io query
        query_parts = [f'name:"{card_name}"']
        if card_number:
            query_parts.append(f'number:{card_number}')
        
        query = ' AND '.join(query_parts)
        print(f"   Query: {query}")
        
        try:
            response = requests.get("https://api.pokemontcg.io/v2/cards", params={
                "q": query,
                "pageSize": 10
            })
            response.raise_for_status()
            
            cards = response.json().get("data", [])
            print(f"   Found {len(cards)} matches in pokemontcg.io:")
            
            if not cards:
                print("❌ No matches found")
                return None
            
            # Show all matches for verification
            for i, card in enumerate(cards):
                name = card.get('name', '')
                number = card.get('number', '')
                set_info = card.get('set', {})
                set_name = set_info.get('name', 'Unknown')
                set_id = set_info.get('id', 'Unknown')
                
                print(f"     {i+1}. {name} #{number} from {set_name} ({set_id})")
                
                # Show image URL for manual verification if needed
                images = card.get('images', {})
                if 'small' in images:
                    print(f"        🖼️ Image: {images['small']}")
            
            # For now, take the first match (most likely correct)
            # In production, you might want to add scoring logic
            best_match = cards[0]
            
            print(f"✅ Selected: {best_match['name']} #{best_match['number']}")
            print(f"   Set: {best_match['set']['name']} ({best_match['set']['id']})")
            
            return best_match
            
        except Exception as e:
            print(f"❌ Error querying pokemontcg.io: {e}")
            return None
    
    def search_price_with_exact_details(self, card_data: Dict) -> Optional[Dict]:
        """
        Step 2: Use EXACT card details for ONE precise PriceTracker search
        
        This is where accuracy matters most!
        """
        name = card_data.get('name')
        number = card_data.get('number') 
        set_info = card_data.get('set', {})
        set_name = set_info.get('name')
        set_id = set_info.get('id')
        
        print(f"\n💰 STEP 2: Getting price for EXACT card:")
        print(f"   Name: {name}")
        print(f"   Number: #{number}")
        print(f"   Set: {set_name} ({set_id})")
        
        # Create multiple search strategies, most precise first
        search_strategies = [
            # Strategy 1: Full name + set ID (most precise)
            {
                "term": f"{name} {set_id}",
                "description": "Name + Set ID"
            },
            
            # Strategy 2: Full name + number  
            {
                "term": f"{name} {number}",
                "description": "Name + Number"
            },
            
            # Strategy 3: Full name + set name
            {
                "term": f"{name} {set_name}",
                "description": "Name + Set Name"
            },
            
            # Strategy 4: Just the exact name (fallback)
            {
                "term": name,
                "description": "Name Only"
            }
        ]
        
        headers = {"Authorization": f"Bearer {self.price_api_key}"}
        
        # Try each strategy until we find a price
        for i, strategy in enumerate(search_strategies):
            search_term = strategy["term"]
            description = strategy["description"]
            
            print(f"\n🎯 Attempt {i+1}: {description}")
            print(f"   Searching: '{search_term}'")
            
            params = {
                "search": search_term,
                "limit": 3,  # Get top 3 results
                "includeHistory": "false"
            }
            
            try:
                response = requests.get(
                    "https://www.pokemonpricetracker.com/api/v2/cards",
                    headers=headers,
                    params=params
                )
                
                # Track API usage
                self.api_calls_made += 1
                remaining = max(0, 200 - self.api_calls_made)
                print(f"   🎫 API Call #{self.api_calls_made} | ~{remaining} remaining")
                
                if response.status_code == 200:
                    data = response.json()
                    results = data.get("data", [])
                    
                    if results:
                        print(f"   ✅ Found {len(results)} results:")
                        
                        for j, result in enumerate(results):
                            result_name = result.get("name", "")
                            result_price = result.get("prices", {}).get("market")
                            result_set = result.get("set", {})
                            result_set_name = result_set.get("name", "Unknown") if isinstance(result_set, dict) else str(result_set)
                            
                            print(f"     {j+1}. {result_name}")
                            print(f"        Set: {result_set_name}")
                            print(f"        Price: ${result_price}")
                        
                        # FIXED: Find the result that matches our exact card name and number
                        target_name = name.lower()
                        target_number = str(number)
                        best_result = None
                        
                        print(f"\n   🔍 Looking for exact match: '{name}' with number #{number}")
                        
                        for result in results:
                            result_name = result.get("name", "").lower()
                            result_name_display = result.get("name", "")
                            
                            # Check if the name matches (contains our target name)
                            name_match = target_name in result_name
                            
                            # Check if the number appears in the result name
                            number_match = (target_number in result_name_display or 
                                          f"#{target_number}" in result_name_display or
                                          f"- {target_number}" in result_name_display)
                            
                            print(f"     Checking: {result_name_display}")
                            print(f"       Name match: {name_match} ('{target_name}' in '{result_name}')")
                            print(f"       Number match: {number_match} ('{target_number}' in name)")
                            
                            if name_match and number_match:
                                best_result = result
                                print(f"     ✅ EXACT MATCH FOUND: {result_name_display}")
                                break
                            elif name_match and not best_result:
                                # Keep as backup if we find name match but no number match
                                best_result = result
                                print(f"     🔶 Name match only (backup): {result_name_display}")
                        
                        if best_result:
                            market_price = best_result.get("prices", {}).get("market")
                            
                            if market_price:
                                match_type = "exact" if (target_name in best_result.get("name", "").lower() and 
                                                       target_number in best_result.get("name", "")) else "name_only"
                                print(f"🎉 SUCCESS: {match_type} match found - ${market_price}")
                                return {
                                    "original_card": card_data,
                                    "price_result": best_result, 
                                    "price": market_price,
                                    "search_strategy": description,
                                    "search_term": search_term,
                                    "match_type": match_type
                                }
                            else:
                                print(f"   ⚠️ Match found but no price available")
                        else:
                            print(f"   ❌ No matching card found in results")
                    else:
                        print(f"   ❌ No results for this search")
                        
                elif response.status_code == 429:
                    print(f"   ❌ Rate limit hit! Used {self.api_calls_made} calls")
                    return None
                else:
                    print(f"   ❌ API error: {response.status_code}")
                    
            except Exception as e:
                print(f"   ❌ Request error: {e}")
        
        print("❌ Could not find price with any search strategy")
        return None
    
    def get_card_price(self, card_input: str) -> Optional[Dict]:
        """
        Complete workflow: Input → pokemontcg.io → PriceTracker → Result
        
        Example: get_card_price("Rayquaza V 194/203")
        """
        print("="*70)
        print(f"🚀 PRECISION SEARCH: {card_input}")
        print("="*70)
        
        # Step 1: Get exact card details (FREE)
        card_data = self.get_exact_card_from_pokemontcg(card_input)
        
        if not card_data:
            print("❌ Failed to find card in pokemontcg.io database")
            return None
        
        # Step 2: Get price using exact details (1 API call)
        price_result = self.search_price_with_exact_details(card_data)
        
        if price_result:
            print("\n" + "="*70)
            print("🎉 SEARCH SUCCESSFUL!")
            print(f"Card: {price_result['original_card']['name']}")
            print(f"Number: #{price_result['original_card']['number']}")
            print(f"Set: {price_result['original_card']['set']['name']}")
            print(f"Price: ${price_result['price']}")
            print(f"Strategy: {price_result['search_strategy']}")
            print(f"API Calls Used: {self.api_calls_made}")
            print("="*70)
            return price_result
        else:
            print(f"❌ Could not find price after {self.api_calls_made} API calls")
            return None

# Simple interface function
def get_card_price(card_input: str) -> Optional[float]:
    """
    Simple function to get just the price
    
    Usage: price = get_card_price("Rayquaza V 194/203")
    """
    pricer = PrecisionCardPricer()
    result = pricer.get_card_price(card_input)
    
    if result:
        return float(result["price"])
    return None

# Usage tracking function
def get_usage_stats():
    """Check how many API calls have been made"""
    # This would need to be tracked globally in a real application
    print("💡 Create a PrecisionCardPricer instance to track usage")

#************************************************************
# TEST YOUR EXAMPLES
#************************************************************
if __name__ == "__main__":
    pricer = PrecisionCardPricer()
    
    # Test 1: Your Rayquaza V example
    print("🧪 TEST 1: Rayquaza V 194/203")
    rayquaza_result = pricer.get_card_price("Rayquaza V 194/203")
    
    print("\n" + "🧪 TEST 2: Cynthia SV82")
    cynthia_result = pricer.get_card_price("Cynthia SV82")
    
    print(f"\n📊 FINAL API USAGE: {pricer.api_calls_made} calls made")
    print(f"Estimated remaining: {max(0, 200 - pricer.api_calls_made)}/200")

🧪 TEST 1: Rayquaza V 194/203
🚀 PRECISION SEARCH: Rayquaza V 194/203
🔍 STEP 1: Finding exact card details for 'Rayquaza V 194/203'
   Parsed - Name: 'Rayquaza V', Number: '194'
   Query: name:"Rayquaza V" AND number:194
   Found 1 matches in pokemontcg.io:
     1. Rayquaza V #194 from Evolving Skies (swsh7)
        🖼️ Image: https://images.pokemontcg.io/swsh7/194.png
✅ Selected: Rayquaza V #194
   Set: Evolving Skies (swsh7)

💰 STEP 2: Getting price for EXACT card:
   Name: Rayquaza V
   Number: #194
   Set: Evolving Skies (swsh7)

🎯 Attempt 1: Name + Set ID
   Searching: 'Rayquaza V swsh7'
   🎫 API Call #1 | ~199 remaining
   ❌ Rate limit hit! Used 1 calls
❌ Could not find price after 1 API calls

🧪 TEST 2: Cynthia SV82
🚀 PRECISION SEARCH: Cynthia SV82
🔍 STEP 1: Finding exact card details for 'Cynthia SV82'
   Parsed - Name: 'Cynthia SV82', Number: 'None'
   Query: name:"Cynthia SV82"
   Found 0 matches in pokemontcg.io:
❌ No matches found
❌ Failed to find card in pokemontcg.io databas